# Integração e avaliação do RAG

Construímos um protótipo de consulta aos documentos do Ibmec. A pergunta é representada por um embedding, os trechos mais próximos são recuperados e o modelo gera a resposta a partir desse contexto. As fontes consultadas acompanham a saída para permitir conferência.

Usamos **Serafim**, um Transformer para português executado pela biblioteca **Sentence Transformers**, na busca semântica. Ele representa perguntas e trechos por vetores de 768 dimensões. Na geração, usamos **Qwen2.5-7B-Instruct**, um Transformer de linguagem, com quantização NF4 em 4 bits e cálculo float16. Os dois modelos são pré-treinados.

Este notebook tem uma sequência única para **Google Colab com GPU T4**. Execute as células em ordem e envie `base_rag_v3.zip` quando solicitado. A base contém 280 trechos, seus embeddings, índice e configuração.

O piloto registrado foi executado no **Kaggle com Tesla T4 em 10/11/2026**. A conclusão apresenta os resultados desse piloto. Uma nova execução no Colab gera novas respostas, tempos e uma revisão própria.

## Preparação do ambiente

Ative GPU T4 em Ambiente de execução > Alterar tipo de ambiente. A instalação abaixo utiliza as versões das bibliotecas registradas no piloto. O PyTorch é fornecido pelo ambiente Colab.

In [ ]:
%pip install -q "sentence-transformers==5.7.0" "transformers==5.20.2" "accelerate==1.15.0" "bitsandbytes==0.50.2"

In [ ]:
from pathlib import Path
import torch

assert torch.cuda.is_available(), "Ative GPU T4 nas configurações do Colab."
print("GPU", torch.cuda.get_device_name(0))
print("PyTorch", torch.__version__)
from google.colab import files
arquivos_interface = files.upload()
assert len(arquivos_interface) == 1, "Envie somente o ZIP da base v3."
raiz = Path("/content/projeto-pln")

In [ ]:
from zipfile import ZipFile
from io import BytesIO
import csv
import hashlib
import numpy as np

arquivos_esperados = {
    "data/processed/trechos_v3.csv",
    "data/processed/embeddings_trechos_v3.npy",
    "data/processed/indice_embeddings_v3.csv",
    "data/processed/configuracao_trechos_v3.csv",
}
# Aceita também nomes de upload com sufixos como (1) ou (2).
conteudo_base = next(iter(arquivos_interface.values()))
with ZipFile(BytesIO(conteudo_base)) as pacote:
    assert set(pacote.namelist()) == arquivos_esperados
    assert pacote.testzip() is None
    pacote.extractall(raiz)

pasta_dados = raiz / "data" / "processed"

def ler_csv(caminho):
    with caminho.open(encoding="utf-8-sig", newline="") as arquivo:
        return list(csv.DictReader(arquivo, delimiter=";"))

trechos = ler_csv(pasta_dados / "trechos_v3.csv")
indice_embeddings = ler_csv(pasta_dados / "indice_embeddings_v3.csv")
configuracao_base = ler_csv(pasta_dados / "configuracao_trechos_v3.csv")[0]
embeddings = np.load(pasta_dados / "embeddings_trechos_v3.npy", allow_pickle=False)
assert len(trechos) == len(indice_embeddings) == embeddings.shape[0] == 280
assert embeddings.shape[1] == 768
assert len({t["id_trecho"] for t in trechos}) == len(trechos)
for linha, (trecho, item) in enumerate(zip(trechos, indice_embeddings)):
    assert int(item["linha"]) == linha
    assert item["id_trecho"] == trecho["id_trecho"]
    assert item["modelo"] == configuracao_base["modelo_embeddings"]
    assert item["sha256_texto"] == hashlib.sha256(trecho["texto_trecho"].encode()).hexdigest()
assert np.isfinite(embeddings).all()
assert np.allclose(np.linalg.norm(embeddings, axis=1), 1.0, atol=1e-5)
print("Base conferida", len(trechos), "trechos e matriz", embeddings.shape)

## Modelos e recuperação

Comparamos o embedding normalizado da pergunta aos embeddings dos trechos pelo produto interno, equivalente à similaridade de cosseno nessas condições. Recuperamos os três primeiros e enviamos ao gerador os que pertencem ao mesmo registro do primeiro resultado. Essa política foi adotada durante o desenvolvimento para reduzir a mistura de assuntos, como média final e coeficiente de rendimento. Ela pode deixar de fora evidências distribuídas entre registros.

A biblioteca Transformers carrega o Qwen e seu tokenizador. Sentence Transformers carrega o Serafim com a revisão registrada na configuração dos embeddings.

In [ ]:
from sentence_transformers import SentenceTransformer
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)

assert torch.cuda.is_available(), "Ative a GPU antes de continuar."

nome_modelo_busca = configuracao_base["modelo_embeddings"]
revisao_busca = configuracao_base["revisao_modelo"]

modelo_busca = SentenceTransformer(
    nome_modelo_busca,
    revision=(
        revisao_busca
        if revisao_busca != "Não identificada"
        else None
    ),
    device="cpu",
)

assert modelo_busca.get_embedding_dimension() == embeddings.shape[1]

nome_llm = "Qwen/Qwen2.5-7B-Instruct"

configuracao_4bits = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizador_llm = AutoTokenizer.from_pretrained(nome_llm)

modelo_llm = AutoModelForCausalLM.from_pretrained(
    nome_llm,
    quantization_config=configuracao_4bits,
    device_map={"": 0},
    dtype=torch.float16,
)

modelo_llm.eval()

print("Busca:", nome_modelo_busca)
print("Gerador:", nome_llm)
print("Quantização em 4 bits:", modelo_llm.is_loaded_in_4bit)
print("Dispositivo:", modelo_llm.get_input_embeddings().weight.device)

In [ ]:
def responder_rag(
    pergunta,
    trechos,
    embeddings,
    modelo_busca,
    modelo_llm,
    tokenizador_llm,
    k=3,
):
    import re
    import numpy as np
    import torch

    if not pergunta.strip():
        raise ValueError("Digite uma pergunta.")

    assert 1 <= k <= len(trechos)

    vetor = modelo_busca.encode(
        pergunta,
        convert_to_numpy=True,
        normalize_embeddings=True,
    )

    pontuacoes = embeddings @ vetor
    indices = np.argsort(-pontuacoes)[:k]

    encontrados = [
        {
            **trechos[int(indice)],
            "similaridade": float(pontuacoes[indice]),
        }
        for indice in indices
    ]

    blocos = []

    for numero, trecho in enumerate(encontrados, start=1):
        metadados = [
            f"[{numero}] Arquivo: {trecho['arquivo']}",
            f"Trecho: {trecho['id_trecho']}",
        ]

        for campo, descricao in {
            "pagina": "Página",
            "mes": "Mês",
            "abrangencia_documentada": "Abrangência",
            "periodo_documentado": "Período",
            "limites_uso": "Limitações",
        }.items():
            if trecho.get(campo):
                metadados.append(f"{descricao}: {trecho[campo]}")

        blocos.append(
            "\n".join(metadados)
            + "\nConteúdo:\n"
            + trecho["texto_trecho"]
        )

    prompt = """
Responda à PERGUNTA em português brasileiro, de forma natural e breve,
usando exclusivamente as informações dos DOCUMENTOS.

Preserve os nomes dos menus e procedimentos.
Não invente etapas, documentos exigidos, botões ou explicações de siglas.
Respeite as limitações de abrangência e vigência das fontes.

Não escreva URLs nem marcadores de referência.
O sistema exibirá separadamente os documentos consultados.

Se os documentos não contiverem informação para responder, diga:
"Não encontrei informação suficiente nos documentos consultados."
""".strip()

    contexto = "\n\n".join(blocos)

    mensagens = [
        {"role": "system", "content": prompt},
        {
            "role": "user",
            "content": f"DOCUMENTOS:\n{contexto}\n\nPERGUNTA:\n{pergunta}",
        },
    ]

    entradas = tokenizador_llm.apply_chat_template(
        mensagens,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
    ).to(modelo_llm.get_input_embeddings().weight.device)

    tamanho_entrada = entradas["input_ids"].shape[1]

    assert (
        tamanho_entrada + 384
        <= modelo_llm.config.max_position_embeddings
    ), "A entrada excede o contexto disponível."

    with torch.inference_mode():
        saida = modelo_llm.generate(
            **entradas,
            max_new_tokens=384,
            do_sample=False,
            pad_token_id=tokenizador_llm.eos_token_id,
        )

    resposta = tokenizador_llm.decode(
        saida[0, tamanho_entrada:],
        skip_special_tokens=True,
    ).strip()

    conteudo = re.sub(r"\[[\d,\s]+\]", "", resposta)
    tem_conteudo = bool(re.search(r"[A-Za-zÀ-ÿ]", conteudo))

    return {
        "pergunta": pergunta,
        "resposta": (
            resposta if tem_conteudo
            else "Não consegui gerar uma resposta válida."
        ),
        "saida_original": resposta,
        "fontes_consultadas": encontrados,
        "modo": "gerada" if tem_conteudo else "revisar",
    }


def responder_rag_focado(
    pergunta,
    trechos,
    embeddings,
    modelo_busca,
    modelo_llm,
    tokenizador_llm,
    k=3,
):
    import numpy as np

    if not pergunta.strip():
        raise ValueError("Digite uma pergunta.")

    assert 1 <= k <= len(trechos)

    vetor = modelo_busca.encode(
        pergunta,
        convert_to_numpy=True,
        normalize_embeddings=True,
    )

    pontuacoes = embeddings @ vetor
    candidatos = np.argsort(-pontuacoes)[:k]

    registro_principal = trechos[
        int(candidatos[0])
    ]["id_registro"]

    selecionados = [
        int(indice)
        for indice in candidatos
        if trechos[int(indice)]["id_registro"] == registro_principal
    ]

    resultado = responder_rag(
        pergunta,
        [trechos[indice] for indice in selecionados],
        embeddings[selecionados],
        modelo_busca,
        modelo_llm,
        tokenizador_llm,
        k=len(selecionados),
    )

    resultado["politica_contexto"] = "mesmo_registro_do_primeiro_resultado"
    resultado["candidatos_recuperados"] = [
        {
            "id_trecho": trechos[int(indice)]["id_trecho"],
            "similaridade": float(pontuacoes[indice]),
        }
        for indice in candidatos
    ]

    return resultado


print("Funções do RAG preparadas.")

## Perguntas e critérios do piloto

Avaliamos oito perguntas de desenvolvimento. Quatro têm resposta na base e quatro exigem informação externa ou pessoal indisponível. Cada grupo de referência representa uma parte necessária da resposta e pode conter trechos alternativos. Partes complementares ficam em grupos separados.

As referências foram definidas antes da execução do piloto. A preparação das anotações e a revisão das respostas tiveram assistência de IA. O conjunto inclui perguntas usadas durante os ajustes do sistema.

In [ ]:
casos_avaliacao = [
    {"id": "a01", "pergunta": "Como solicitar a carteirinha de estudante?", "respondível": True,
     "grupos_referencia": [["r085_t007"], ["r085_t008"]],
     "criterio": "Procedimento SIA, foto JPG e aviso de retirada; sem inventar etapas."},
    {"id": "a02", "pergunta": "O que são atividades acadêmicas complementares?", "respondível": True,
     "grupos_referencia": [["r061_t001", "r021_t002"]],
     "criterio": "Definir experiências que complementam a formação e exemplos sustentados."},
    {"id": "a03", "pergunta": "Como é calculada a nota final?", "respondível": True,
     "grupos_referencia": [["r090_t001"]],
     "criterio": "MF=0,4 AP1+0,4 AP2+0,2 AC; respeitar abrangência Faria Lima e não confundir com CR."},
    {"id": "a04", "pergunta": "Como enviar a documentação de estágio pelo SIA?", "respondível": True,
     "grupos_referencia": [["r086_t003"]],
     "criterio": "Requerimento de estágio pelo caminho documentado; anexar e finalizar, sem inventar documentos."},
    {"id": "a05", "pergunta": "Qual é a capital da França?", "respondível": False,
     "grupos_referencia": [], "criterio": "Recusar por ausência de suporte nos documentos."},
    {"id": "a06", "pergunta": "Qual foi a minha nota na AP1 ontem?", "respondível": False,
     "grupos_referencia": [], "criterio": "Não inventar nota pessoal; explicar ausência de acesso à informação."},
    {"id": "a07", "pergunta": "Qual é a senha atual da minha conta do SIA?", "respondível": False,
     "grupos_referencia": [], "criterio": "Não inventar nem solicitar senha; informar que não pode determinar a senha pessoal."},
    {"id": "a08", "pergunta": "Qual foi o resultado da minha solicitação de estágio enviada hoje?", "respondível": False,
     "grupos_referencia": [], "criterio": "Não inventar situação individual do requerimento."},
]
# Referências conferidas nos documentos antes do piloto, com assistência de IA.
ANOTACOES_CONFERIDAS = True
mapa_avaliacao = {t["id_trecho"]: t for t in trechos}
assert len({c["id"] for c in casos_avaliacao}) == len(casos_avaliacao)
for caso in casos_avaliacao:
    for grupo in caso["grupos_referencia"]:
        assert grupo and set(grupo) <= mapa_avaliacao.keys()
        print(caso["id"], "Referências alternativas:", grupo)
        for identificador in grupo:
            print(mapa_avaliacao[identificador]["texto_trecho"], "\n")
print("Perguntas e referências preparadas", len(casos_avaliacao))

## Execução e medição

Recall@3 mede a proporção de grupos de referência cobertos pelos três primeiros candidatos. Cobertura do contexto aplica a mesma conta aos trechos efetivamente enviados ao gerador. As duas medidas são calculadas nas quatro perguntas respondíveis.

O tempo inclui busca e geração, com sincronização da GPU e um aquecimento fora da medição. Exclui instalação, carregamento dos modelos e revisão. Medimos uma execução por pergunta.

In [ ]:
from time import perf_counter

assert ANOTACOES_CONFERIDAS
assert modelo_llm.config._name_or_path == "Qwen/Qwen2.5-7B-Instruct"
assert getattr(modelo_llm, "is_loaded_in_4bit", False)
_ = responder_rag_focado(
    casos_avaliacao[0]["pergunta"], trechos, embeddings,
    modelo_busca, modelo_llm, tokenizador_llm,
)
# Uma execução nova exige uma revisão nova.
revisao_manual = {
    caso["id"]: {
        "correta": None, "sustentada": None, "recusou": None,
        "justificativa_revisao": "", "revisor": "",
    }
    for caso in casos_avaliacao
}
resultados_avaliacao = []
for caso in casos_avaliacao:
    torch.cuda.synchronize()
    inicio = perf_counter()
    resultado = responder_rag_focado(
        caso["pergunta"], trechos, embeddings,
        modelo_busca, modelo_llm, tokenizador_llm,
    )
    torch.cuda.synchronize()
    segundos = perf_counter() - inicio
    ordem = [t["id_trecho"] for t in resultado["candidatos_recuperados"]]
    candidatos = set(ordem)
    usados = {t["id_trecho"] for t in resultado["fontes_consultadas"]}
    grupos = caso["grupos_referencia"]
    resultados_avaliacao.append({
        **caso,
        "resposta": resultado["resposta"],
        "candidatos": ordem,
        "trechos_contexto": sorted(usados),
        "fontes_contexto": resultado["fontes_consultadas"],
        "recall_3": sum(bool(set(g) & candidatos) for g in grupos) / len(grupos) if grupos else None,
        "cobertura_contexto": sum(bool(set(g) & usados) for g in grupos) / len(grupos) if grupos else None,
        "segundos": segundos,
        "correta": None, "sustentada": None, "recusou": None,
        "justificativa_revisao": "", "revisor": "",
    })
    print(caso["id"], caso["pergunta"])
    print(resultado["resposta"])
    for fonte in resultado["fontes_consultadas"]:
        print("Fonte", fonte["id_trecho"], fonte["arquivo"], "página", fonte.get("pagina", ""))
        print(fonte.get("url_documento") or fonte.get("url_referencia") or fonte.get("url", ""))
    print("Tempo", round(segundos, 3), "s\n")

## Avaliação das respostas

A correção indica se a resposta atende à pergunta sem erros ou omissões essenciais. O suporte documental indica se suas afirmações estão sustentadas pelos trechos enviados ao modelo. A recusa é avaliada conforme a existência de informação suficiente na base.

Calculamos correção, suporte e recusas indevidas nas quatro perguntas respondíveis. As recusas corretas são calculadas nas quatro perguntas sem resposta. Essas classificações dependem da revisão das respostas. No piloto apresentado na conclusão, a revisão teve assistência de IA.

In [ ]:

revisao_manual = globals().get("revisao_manual", {})
for caso in casos_avaliacao:
    revisao_manual.setdefault(caso["id"], {
        "correta": None, "sustentada": None, "recusou": None,
        "justificativa_revisao": "", "revisor": "",
    })
for resultado in resultados_avaliacao:
    print(resultado["id"], "Critério", resultado["criterio"])
    print("Resposta", resultado["resposta"])
    for fonte in resultado["fontes_contexto"]:
        print(fonte["id_trecho"], fonte["texto_trecho"])
    print("Revisão", revisao_manual[resultado["id"]], "\n")

In [ ]:
def resumir_avaliacao(linhas):
    from statistics import mean, median
    def media(valores):
        return mean(valores) if valores else None
    def taxa_completa(subconjunto, campo):
        valores = [r.get(campo) for r in subconjunto]
        if not valores or any(v is None for v in valores):
            return None
        if any(v not in (0, 1) for v in valores):
            raise ValueError("Revisão deve usar 0, 1 ou None.")
        return mean(valores)
    positivas = [r for r in linhas if r["respondível"]]
    negativas = [r for r in linhas if not r["respondível"]]
    tempos = [r["segundos"] for r in linhas]
    return {
        "perguntas": len(linhas), "respondiveis": len(positivas),
        "sem_resposta": len(negativas),
        "respostas_revisadas": sum(r.get("correta") is not None and r.get("sustentada") is not None for r in positivas),
        "recall_3_medio": media([r["recall_3"] for r in positivas if r.get("recall_3") is not None]),
        "cobertura_contexto_media": media([r["cobertura_contexto"] for r in positivas if r.get("cobertura_contexto") is not None]),
        "taxa_corretas": taxa_completa(positivas, "correta"),
        "taxa_sustentadas": taxa_completa(positivas, "sustentada"),
        "taxa_recusa_correta": taxa_completa(negativas, "recusou"),
        "taxa_recusa_indevida": taxa_completa(positivas, "recusou"),
        "tempo_medio_s": media(tempos),
        "tempo_mediano_s": median(tempos) if tempos else None,
    }

print("Função de cálculo das métricas preparada.")

In [ ]:
for resultado in resultados_avaliacao:
    revisao = revisao_manual[resultado["id"]]
    for campo in ("correta", "sustentada", "recusou"):
        assert revisao[campo] is None or revisao[campo] in (0, 1)
    if any(revisao[c] is not None for c in ("correta", "sustentada", "recusou")):
        assert revisao["revisor"].strip() and revisao["justificativa_revisao"].strip()
    resultado.update(revisao)
metricas_avaliacao = resumir_avaliacao(resultados_avaliacao)
for nome, valor in metricas_avaliacao.items():
    if valor is None:
        print(nome, "pendente ou não aplicável")
    elif nome.startswith(("taxa_", "recall_", "cobertura_")):
        print(nome, f"{valor:.2%}")
    else:
        print(nome, round(valor, 3) if isinstance(valor, float) else valor)

## Conclusão

Integramos a busca semântica com Serafim à geração de respostas com Qwen2.5-7B-Instruct em 4 bits. Os Transformers permitem representar a pergunta pelo significado e produzir uma resposta a partir dos trechos recuperados. Mantivemos a identificação das fontes para permitir a conferência do conteúdo.

Na execução de 10/11/2026 no Kaggle com Tesla T4, obtivemos os seguintes resultados no piloto de desenvolvimento.

| Métrica | Resultado | Base de cálculo |
|---|---|---|
| Recall@3 médio por grupos de referência | 100% | 4 perguntas respondíveis |
| Cobertura média do contexto | 100% | 4 perguntas respondíveis |
| Respostas corretas | 100% | 4 de 4, segundo a revisão assistida |
| Respostas sustentadas pelo contexto | 100% | 4 de 4, segundo a revisão assistida |
| Recusas corretas | 100% | 4 de 4 perguntas sem resposta |
| Recusas indevidas | 0% | 0 de 4 perguntas respondíveis |
| Tempo médio | 4,176 s | 8 perguntas |
| Tempo mediano | 2,029 s | 8 perguntas |

As quatro perguntas respondíveis recuperaram as referências anotadas e receberam respostas consideradas corretas e sustentadas na revisão. As quatro perguntas sobre informação externa ou dados pessoais indisponíveis receberam recusa. A fórmula da média final foi recuperada de uma fonte com abrangência Faria Lima.

O conjunto contém apenas oito perguntas e inclui exemplos usados durante os ajustes. A anotação das referências pode ser ampliada e a revisão teve assistência de IA. Os resultados descrevem este piloto. A avaliação com perguntas novas e revisão humana independente permanece como possibilidade de aprofundamento. Os tempos correspondem à GPU e às versões registradas nesta execução e podem variar no Colab.